# Building LLM

## Goal

This notebook is a hands-on journey to build a language model from scratch.

Each version introduces one new concept, allowing the model to evolve step by step while practicing language-model development.

---

## Version 1

In this version, we build a Character Statistical Language Model with standard Python.

The model reads an English training text, creates adjacent character pairs, counts character transitions, converts the counts into probabilities, and generates new text one character at a time.

This version keeps the full language-model flow simple and visible.

## 1. Imports

In [1]:
import random
from collections import Counter, defaultdict

## 2. Training Data

### Training text

The English corpus is included directly in the notebook. It provides the text from which the model learns character transitions.

In [2]:
corpus = 'language models learn patterns from text.\na small model predicts what character may come next.\nwe begin with counting because counting is easy to inspect.\nthe model sees letters, spaces, and punctuation.\neach prediction comes from examples found in the training text.\nsimple systems help us understand more advanced systems.\nlater versions will learn parameters with neural networks.\nclear experiments make machine learning easier to study.'

print(corpus)
print("Characters:", len(corpus))

language models learn patterns from text.
a small model predicts what character may come next.
we begin with counting because counting is easy to inspect.
the model sees letters, spaces, and punctuation.
each prediction comes from examples found in the training text.
simple systems help us understand more advanced systems.
later versions will learn parameters with neural networks.
clear experiments make machine learning easier to study.
Characters: 440


### Vocabulary

The vocabulary is the set of symbols the model can represent. Because this is a character model, every letter, space, punctuation mark and newline is a token.

In [3]:
vocabulary = sorted(set(corpus))

print("Vocabulary size:", len(vocabulary))
print("Vocabulary:", repr("".join(vocabulary)))

Vocabulary size: 27
Vocabulary: '\n ,.abcdefghiklmnoprstuvwxy'


### Character pairs

Each pair of adjacent characters is one training example.

For the text `model` the examples are:

- `m -> o`
- `o -> d`
- `d -> e`
- `e -> l`

The current character is the context. The following character is the target.

In [4]:
examples = list(zip(corpus, corpus[1:]))

print("Number of examples:", len(examples))
print("First 12 examples:", examples[:12])

Number of examples: 439
First 12 examples: [('l', 'a'), ('a', 'n'), ('n', 'g'), ('g', 'u'), ('u', 'a'), ('a', 'g'), ('g', 'e'), ('e', ' '), (' ', 'm'), ('m', 'o'), ('o', 'd'), ('d', 'e')]


## 3. Statistical Model

### Transition counts

For every context character, the model counts the characters that followed it in the corpus.

This table of counts is the complete trained model.

In [5]:
def train_model(text):
    counts = defaultdict(Counter)

    for current_character, next_character in zip(text, text[1:]):
        counts[current_character][next_character] += 1

    return counts

In [6]:
model = train_model(corpus)

print("What followed 'm':", model["m"])
print("What followed a space:", model[" "].most_common(10))

What followed 'm': Counter({'o': 4, 'a': 4, 'e': 4, ' ': 2, 'p': 2, 's': 2})
What followed a space: [('m', 7), ('t', 6), ('s', 6), ('p', 5), ('c', 5), ('l', 4), ('w', 4), ('e', 4), ('f', 3), ('n', 3)]


### Probabilities

A probability is a count divided by the total count for the same context.

All next-character probabilities for one context add up to 1.

In [7]:
def next_character_probabilities(model, current_character):
    next_counts = model.get(current_character)

    if not next_counts:
        return {}

    total = sum(next_counts.values())
    return {character: count / total for character, count in next_counts.items()}

In [8]:
probabilities_after_m = next_character_probabilities(model, "m")
print(probabilities_after_m)
print("Total:", sum(probabilities_after_m.values()))

{'o': 0.2222222222222222, ' ': 0.1111111111111111, 'a': 0.2222222222222222, 'e': 0.2222222222222222, 'p': 0.1111111111111111, 's': 0.1111111111111111}
Total: 1.0


## 4. Generator

### Sample the next character

Sampling uses the probability distribution learned from the training text. Characters with higher probabilities are more likely to be selected, while other possible characters can still be chosen.

In [9]:
def sample_next_character(model, current_character, random_generator):
    probabilities = next_character_probabilities(model, current_character)

    if not probabilities:
        return None

    characters = list(probabilities)
    weights = list(probabilities.values())
    return random_generator.choices(characters, weights=weights, k=1)[0]

In [10]:
demo_random = random.Random(42)
print("Five samples after 'm':", [sample_next_character(model, "m", demo_random) for _ in range(5)])

Five samples after 'm': ['e', 'o', ' ', ' ', 'e']


### Generate text

Generation repeats the same loop:

1. read the current character;
2. sample the next character;
3. append it to the output;
4. use the new character as the next context.

The seed makes the example reproducible.

In [11]:
def generate_text(model, start_character="i", length=200, seed=42):
    if length < 1:
        raise ValueError("Length must be at least 1.")

    if start_character not in model:
        raise ValueError("The start character was not seen during training.")

    random_generator = random.Random(seed)
    generated = [start_character]
    current_character = start_character

    for _ in range(length - 1):
        next_character = sample_next_character(model, current_character, random_generator)

        if next_character is None:
            break

        generated.append(next_character)
        current_character = next_character

    return "".join(generated)

In [12]:
generated_text = generate_text(model, start_character="i", length=300, seed=42)
print(generated_text)

is paystern s pun wicachadicto ioudito exan frodext.
wo ctins.
edvelarel eachin pell tene mo ingierimont puat.
cts am cod witte amprncla mectsy erous fo sy pr pantetuar pad siomouneg vat.
weraroramanimstua prns.
lerier leunecth e uar ctetexath hay tin berie paimprnco ve intrngun comeletoding ctit.
a


## 5. Tests

These assertions verify the main parts of the model.

In [13]:
assert len(examples) == len(corpus) - 1
assert abs(sum(next_character_probabilities(model, "m").values()) - 1.0) < 1e-12
assert generate_text(model, "l", 30, seed=10) == generate_text(model, "l", 30, seed=10)
assert len(generate_text(model, "l", 30, seed=10)) == 30

print("All checks passed.")

All checks passed.


## Notes

- The training text provides the examples used by the model.
- Character pairs represent the current context and the next-character target.
- Transition counts are the learned statistical model.
- Probabilities describe the possible next characters for each context.
- The generator repeatedly samples the next character to create new text.
- The tests verify the main components and the reproducibility of generation.

Future versions will introduce new components and gradually evolve the architecture.